## Assignment: Object detection — Two-Stage Model (Faster R-CNN)
- Alumno 1:
- Alumno 2:
- Alumno 3:

Model: **Faster R-CNN** with ResNet50 backbone (two-stage detector)
- Stage 1: Region Proposal Network (RPN) — proposes candidate regions
- Stage 2: ROI Align + classification head — classifies proposals

Framework: TensorFlow Object Detection API

In [1]:
import os
# Must be set BEFORE any protobuf/TF import to avoid Descriptor compatibility errors
# with protobuf >= 4.x and old TF OD API generated _pb2.py files
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TF {tf.__version__} | GPUs: {[g.name for g in gpus] or 'none (using CPU)'}")

2026-04-04 22:58:40.716428: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775343520.910195      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775343520.969078      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775343521.463669      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775343521.463709      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775343521.463711      23 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: ['/physical_device:GPU:0', '/physical_device:GPU:1']


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import requests
import zipfile
import json

url = 'https://drive.upm.es/s/P7nEf3Bygns7tbM/download'
zip_name = 'dataset.zip'
target_file = 'xview_det_train.json'
found_path = None
search_roots = ['.', '..', '../..']

for search_root in search_roots:
    if not os.path.exists(search_root):
        continue
    for root, dirs, files in os.walk(search_root):
        if target_file in files:
            found_path = os.path.abspath(os.path.join(root, target_file))
            break
    if found_path:
        break

if not found_path:
    if not os.path.exists(zip_name):
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        with open(zip_name, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    if (not os.path.exists(zip_name)) or os.path.getsize(zip_name) < 10000:
        raise FileNotFoundError(f"Le fichier {zip_name} est absent ou trop petit.")
    else:
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall(".")

        for search_root in search_roots:
            if not os.path.exists(search_root):
                continue
            for root, dirs, files in os.walk(search_root):
                if target_file in files:
                    found_path = os.path.abspath(os.path.join(root, target_file))
                    break
            if found_path:
                break

if not found_path:
    raise FileNotFoundError(f"{target_file} reste introuvable apres extraction.")

print(f"SUCCES : Fichier trouve a : {found_path}")
with open(found_path) as ifs:
    json_data = json.load(ifs)
print("Base de donnees chargee avec succes !")

SUCCES : Fichier trouve a : /kaggle/working/xview_det_train.json


Base de donnees chargee avec succes !


In [3]:
import uuid
import numpy as np

class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([])

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [4]:
categories = {0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'}

In [5]:
import warnings
import rasterio
import numpy as np
import os

def resolve_image_path(filename):
    candidates = []
    if os.path.isabs(filename):
        candidates.append(filename)
    else:
        candidates.extend([
            filename,
            os.path.join(BASE_DIR, filename) if ('BASE_DIR' in globals() and BASE_DIR is not None) else filename,
            os.path.join('.', filename),
            os.path.join('..', filename),
            os.path.join('../..', filename),
        ])
    seen = set()
    for path in candidates:
        norm = os.path.abspath(path)
        if norm in seen:
            continue
        seen.add(norm)
        if os.path.exists(norm):
            return norm
    raise FileNotFoundError(f'Image introuvable: {filename}')

def load_geoimage(filename):
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    full_path = resolve_image_path(filename)
    with rasterio.open(full_path, 'r') as src:
        img = np.zeros((src.height, src.width, src.count), dtype=np.float32)
        for b in range(src.count):
            band = src.read(b + 1).astype(np.float32)
            bmin, bmax = band.min(), band.max()
            img[:, :, b] = (band - bmin) / (bmax - bmin + 1e-6) * 255.0
    return img.astype(np.uint8)

#### Training
Design and train a detector to deal with the “xview_detection” perception task.

In [6]:
import json
import os

json_file = found_path if ('found_path' in globals() and found_path is not None) else '../PROJECT/xview_detection/xview_det_train.json'
json_file = os.path.abspath(json_file)
if not os.path.exists(json_file):
    raise FileNotFoundError(f"Le fichier json de dataset est introuvable: {json_file}")

with open(json_file) as ifs:
    json_data = json.load(ifs)

BASE_DIR = os.path.dirname(json_file)
print("BASE_DIR:", BASE_DIR)

BASE_DIR: /kaggle/working


In [7]:
import numpy as np
from collections import defaultdict

ann_index = defaultdict(list)
for ann_elem in json_data['annotations'].values():
    ann_index[ann_elem['image_id']].append(ann_elem)

counts = dict.fromkeys(categories.values(), 0)
anns = []
for json_img in json_data['images'].values():
    image = GenericImage(json_img['filename'])
    image.tile = np.array([0, 0, json_img['width'], json_img['height']])
    for json_ann in ann_index[json_img['image_id']]:
        obj = GenericObject()
        obj.id = json_ann['image_id']
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]),
                  int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        cat_id = json_ann['category_id']
        obj.category = categories[cat_id] if cat_id in categories else cat_id
        counts[obj.category] += 1
        image.add_object(obj)
    anns.append(image)
print(counts)

{'Small car': 188300, 'Bus': 6269, 'Truck': 10600, 'Building': 275943}


In [8]:
from sklearn.model_selection import train_test_split

if len(anns) < 2:
    raise ValueError('Le dataset doit contenir au moins 2 images pour faire train/validation.')

anns_train, anns_valid = train_test_split(anns, test_size=0.1, random_state=1, shuffle=True)

bus_truck_anns = [a for a in anns_train if any(
    o.category in ['Bus', 'Truck'] for o in a.objects
)]
anns_train = anns_train + bus_truck_anns

print('Number of training images: ' + str(len(anns_train)))
print('Number of validation images: ' + str(len(anns_valid)))
print(f'Bus/Truck images dupliquees : {len(bus_truck_anns)} ajoutees au train set')

Number of training images: 10117
Number of validation images: 761
Bus/Truck images dupliquees : 3272 ajoutees au train set


In [9]:
import os, sys, tarfile, urllib.request, subprocess, importlib, tensorflow as tf
# GPU memory growth — must be uniform across all devices before TF initializes
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for _gpu in gpus:
      tf.config.experimental.set_memory_growth(_gpu, True)
  except RuntimeError as _e:
    print(f'GPU memory growth: {_e}')
                                                                                                                             
                                                                                                                                                                                                           
TF_MODELS_DIR = '/tmp/tf_models'                                                                                                                                                                             
RESEARCH_DIR = os.path.join(TF_MODELS_DIR, 'research')                                                                                                                                                       
                                                                                                                                                                                                           
                                                                                                                                                                                                           
def patch_tfod_keras_compatibility():                                                                                                                                                                        
  patch_file = os.path.join(                                                                                                                                                                               
      RESEARCH_DIR, 'object_detection', 'core', 'freezable_sync_batch_norm.py'                                                                                                                             
  )                                                                                                                                                                                                        
  if not os.path.exists(patch_file):                                                                                                                                                                       
      return                                                                                                                                                                                               
  with open(patch_file, 'r', encoding='utf-8') as f:                                                                                                                                                       
      src = f.read()                                                                                                                                                                                      
  old = 'tf.keras.layers.experimental.SyncBatchNormalization'                                                                                                                                              
  new = 'getattr(tf.keras.layers, "SyncBatchNormalization", tf.keras.layers.BatchNormalization)'                                                                                                           
  if old in src:                                                                                                                                                                                           
      with open(patch_file, 'w', encoding='utf-8') as f:                                                                                                                                                   
          f.write(src.replace(old, new))                                                                                                                                                                   
      print('  Patched freezable_sync_batch_norm.')                                                                                                                                                        
                                                                                                                                                                                                           
                                                                                                                                                                                                           
def patch_tfod_visualization_compatibility():                                                                                                                                                                
  patch_file = os.path.join(                                                                                                                                                                               
      RESEARCH_DIR, 'object_detection', 'utils', 'visualization_utils.py'                                                                                                                                  
  )                                                                                                                                                                                                        
  if not os.path.exists(patch_file):                                                                                                                                                                       
      return                                                                                                                                                                                               
  with open(patch_file, 'r', encoding='utf-8') as f:                                                                                                                                                       
      src = f.read()                                                                                                                                                                                      
  old_block = (                                                                                                                                                                                            
      'import PIL.Image as Image\n'                                                                                                                                                                        
      'import PIL.ImageColor as ImageColor\n'                                                                                                                                                              
      'import PIL.ImageDraw as ImageDraw\n'                                                                                                                                                                
      'import PIL.ImageFont as ImageFont\n'                                                                                                                                                                
  )                                                                                                                                                                                                        
  new_block = (                                                                                                                                                                                            
      'try:\n'                                                                                                                                                                                             
      '  import PIL.Image as Image\n'                                                                                                                                                                      
      '  import PIL.ImageColor as ImageColor\n'                                                                                                                                                            
      '  import PIL.ImageDraw as ImageDraw\n'                                                                                                                                                              
      '  import PIL.ImageFont as ImageFont\n'                                                                                                                                                              
      'except Exception:\n'                                                                                                                                                                                
      '  Image = ImageColor = ImageDraw = ImageFont = None\n'                                                                                                                                              
  )                                                                                                                                                                                                        
  if old_block in src:                                                                                                                                                                                     
      with open(patch_file, 'w', encoding='utf-8') as f:                                                                                                                                                   
          f.write(src.replace(old_block, new_block))                                                                                                                                                       
      print('  Patched visualization_utils.')                                                                                                                                                              
                                                                                                                                                                                                           
                                                                                                                                                                                                           
def patch_tfod_model_builder_optional_imports():                                                                                                                                                             
  patch_file = os.path.join(                                                                                                                                                                               
      RESEARCH_DIR, 'object_detection', 'builders', 'model_builder.py'                                                                                                                                     
  )                                                                                                                                                                                                       
  if not os.path.exists(patch_file):                                                                                                                                                                       
      return                                                                                                                                                                                               
  with open(patch_file, 'r', encoding='utf-8') as f:                                                                                                                                                       
      src = f.read()                                                                                                                                                                                       
  old_import = (                                                                                                                                                                                           
      '    from object_detection.models import '                                                                                                                                                           
      'ssd_efficientnet_bifpn_feature_extractor as ssd_efficientnet_bifpn\n'                                                                                                                               
  )                                                                                                                                                                                                        
  new_import = (                                                                                                                                                                                           
      '    try:\n'                                                                                                                                                                                         
      '      from object_detection.models import '                                                                                                                                                         
      'ssd_efficientnet_bifpn_feature_extractor as ssd_efficientnet_bifpn\n'                                                                                                                               
      '    except Exception:\n'                                                                                                                                                                            
      '      ssd_efficientnet_bifpn = None\n'                                                                                                                                                              
  )                                                                                                                                                                                                        
  if old_import in src:                                                                                                                                                                                    
      src = src.replace(old_import, new_import)                                                                                                                                                            
  for level in range(8):                                                                                                                                                                                   
      class_name = f'SSDEfficientNetB{level}BiFPNKerasFeatureExtractor'                                                                                                                                    
      block = (                                                                                                                                                                                            
          f"      'ssd_efficientnet-b{level}_bifpn_keras':\n"                                                                                                                                              
          f"          ssd_efficientnet_bifpn.{class_name},\n"                                                                                                                                              
      )                                                                                                                                                                                                    
      src = src.replace(block, '')                                                                                                                                                                         
  with open(patch_file, 'w', encoding='utf-8') as f:                                                                                                                                                       
      f.write(src)                                                                                                                                                                                         
  print('  Patched model_builder.')                                                                                                                                                                        
                                                                                                                                                                                                           
                                                                                                                                                                                                           
def patch_tfod_resnet_layers_compat():                                                                                                                                                                      
  patch_file = os.path.join(                                                                                                                                                                               
      RESEARCH_DIR, 'object_detection', 'models', 'keras_models', 'resnet_v1.py'                                                                                                                           
  )                                                                                                                                                                                                        
  if not os.path.exists(patch_file):                                                                                                                                                                       
      return                                                                                                                                                                                               
  with open(patch_file, 'r', encoding='utf-8') as f:                                                                                                                                                       
      src = f.read()                                                                                                                                                                                       
  changed = False                                                                                                                                                                                          
  for variant in ['ResNet50', 'ResNet101', 'ResNet152']:                                                                                                                                                   
      old = (                                                                                                                                                                                              
          f'  return tf.keras.applications.resnet.{variant}(\n'                                                                                                                                            
          f'      layers=layers_override, **kwargs)\n'                                                                                                                                                     
      )                                                                                                                                                                                                    
      new = (                                                                                                                                                                                              
          f'  try:\n'                                                                                                                                                                                      
          f'    return tf.keras.applications.resnet.{variant}(\n'                                                                                                                                          
          f'        layers=layers_override, **kwargs)\n'                                                                                                                                                   
          f'  except TypeError:\n'                                                                                                                                                                         
          f'    return tf.keras.applications.resnet.{variant}(**kwargs)\n'                                                                                                                                 
      )                                                                                                                                                                                                    
      if old in src:                                                                                                                                                                                       
          src = src.replace(old, new)                                                                                                                                                                      
          changed = True                                                                                                                                                                                   
  if changed:                                                                                                                                                                                             
      with open(patch_file, 'w', encoding='utf-8') as f:
          f.write(src)                                                                                                                                                                                     
      print('  Patched resnet_v1.py (layers= kwarg).')                                                                                                                                                     
                                                                                                                                                                                                           
                                                                                                                                                                                                           
def patch_tfod_static_shape_compat():                                                                                                                                                                        
  patch_file = os.path.join(                                                                                                                                                                               
      RESEARCH_DIR, 'object_detection', 'utils', 'static_shape.py'                                                                                                                                         
  )                                                                                                                                                                                                       
  if not os.path.exists(patch_file):                                                                                                                                                                       
      return                                                                                                                                                                                               
  with open(patch_file, 'r', encoding='utf-8') as f:                                                                                                                                                       
      src = f.read()                                                                                                                                                                                       
  old = '  tensor_shape.assert_has_rank('                                                                                                                                                                  
  new = (                                                                                                                                                                                                  
      '  if not hasattr(tensor_shape, "assert_has_rank"):\n'                                                                                                                                               
      '    tensor_shape = tf.TensorShape(tensor_shape)\n'                                                                                                                                                  
      '  tensor_shape.assert_has_rank('                                                                                                                                                                    
  )                                                                                                                                                                                                        
  if old in src:                                                                                                                                                                                           
      src = src.replace(old, new)                                                                                                                                                                          
      with open(patch_file, 'w', encoding='utf-8') as f:                                                                                                                                                   
          f.write(src)                                                                                                                                                                                     
      print('  Patched static_shape.py (tuple TensorShape compat).')                                                                                                                                       
                                                                                                                                                                                                           
                                                                                                                                                                                                           
def patch_tfod_model_util_compat():                                                                                                                                                                          
  """Replace .experimental_ref() with _tref() in model_util.py (removed in Keras 3)."""                                                                                                                    
  import re                                                                                                                                                                                               
  patch_file = os.path.join(                                                                                                                                                                               
      RESEARCH_DIR, 'object_detection', 'utils', 'model_util.py'                                                                                                                                           
  )                                                                                                                                                                                                        
  if not os.path.exists(patch_file):                                                                                                                                                                       
      return                                                                                                                                                                                               
  with open(patch_file, 'r', encoding='utf-8') as f:                                                                                                                                                       
      src = f.read()                                                                                                                                                                                       
  if '.experimental_ref()' not in src:                                                                                                                                                                     
      return                                                                                                                                                                                               
  helper = (                                                                                                                                                                                               
      '\n\n_TREF_CACHE = {}\n\n'                                                                                                                                                                          
      'def _tref(t):\n'                                                                                                                                                                                    
      '  fn = getattr(t, "ref", None) or getattr(t, "experimental_ref", None)\n'                                                                                                                           
      '  if fn:\n'                                                                                                                                                                                         
      '    return fn()\n'                                                                                                                                                                                  
      '  k = id(t)\n'                                                                                                                                                                                      
      '  _TREF_CACHE[k] = t\n'                                                                                                                                                                             
      '  return k\n\n'                                                                                                                                                                                     
  )                                                                                                                                                                                                        
  new_src = re.sub(r'(\w+)\.experimental_ref\(\)', r'_tref(\1)', src)                                                                                                                                      
  insert_pos = new_src.find('\ndef ')                                                                                                                                                                      
  if insert_pos < 0:                                                                                                                                                                                       
      insert_pos = 0                                                                                                                                                                                       
  new_src = new_src[:insert_pos] + helper + new_src[insert_pos:]                                                                                                                                           
  with open(patch_file, 'w', encoding='utf-8') as f:                                                                                                                                                       
      f.write(new_src)                                                                                                                                                                                     
  print('  Patched model_util.py (experimental_ref → _tref).')                                                                                                                                             
                                                                                                                                                                                                           
                                                                                                                                                                                                           
# ── Installation ───────────────────────────────────────────────────────                                                                                                                                    
if RESEARCH_DIR not in sys.path:                                                                                                                                                                             
  sys.path.insert(0, RESEARCH_DIR)                                                                                                                                                                         
importlib.invalidate_caches()                                                                                                                                                                                
                                                                                                                                                                                                           
try:                                                                                                                                                                                                         
  from object_detection.builders import model_builder                                                                                                                                                     
  from object_detection.utils import config_util                                                                                                                                                           
  print('TF Object Detection API already available.')                                                                                                                                                      
except Exception as first_err:                                                                                                                                                                               
  print(f'Preparing TF Object Detection API ({type(first_err).__name__}: {first_err})')                                                                                                                    
  if not os.path.exists(TF_MODELS_DIR):                                                                                                                                                                    
      subprocess.run([                                                                                                                                                                                     
          'git', 'clone', '--depth', '1',                                                                                                                                                                  
          'https://github.com/tensorflow/models', TF_MODELS_DIR                                                                                                                                            
      ], check=True, capture_output=True)                                                                                                                                                                  
      print('  Repo cloned.')                                                                                                                                                                              
  subprocess.run(['apt-get', 'install', '-y', '-q', 'protobuf-compiler'], check=True)                                                                                                                      
  subprocess.run(                                                                                                                                                                                          
      'protoc object_detection/protos/*.proto --python_out=.',                                                                                                                                             
      shell=True, cwd=RESEARCH_DIR, check=True                                                                                                                                                             
  )                                                                                                                                                                                                        
  print('  Protos compiled.')                                                                                                                                                                              
  subprocess.run(                                                                                                                                                                                          
      ['cp', 'object_detection/packages/tf2/setup.py', '.'],                                                                                                                                               
      cwd=RESEARCH_DIR, check=True                                                                                                                                                                         
  )                                                                                                                                                                                                        
  subprocess.run(                                                                                                                                                                                          
      [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],                                                                                                                                           
      cwd=RESEARCH_DIR, check=True                                                                                                                                                                         
  )                                                                                                                                                                                                        
  print('  Package installed.')                                                                                                                                                                            
  patch_tfod_keras_compatibility()                                                                                                                                                                         
  patch_tfod_visualization_compatibility()                                                                                                                                                                 
  patch_tfod_model_builder_optional_imports()                                                                                                                                                              
  patch_tfod_resnet_layers_compat()                                                                                                                                                                        
  patch_tfod_static_shape_compat()                                                                                                                                                                         
  patch_tfod_model_util_compat()                                                                                                                                                                           
  importlib.invalidate_caches()                                                                                                                                                                            
  from object_detection.builders import model_builder                                                                                                                                                      
  from object_detection.utils import config_util                                                                                                                                                           
  print('TF Object Detection API ready.')                                                                                                                                                                  
                                                                                                                                                                                                           
# ── Runtime patches (run every time) ──────────────────────────────────                                                                                                                                     
                                                                                                                                                                                                           
# 1. tf.logging removed in TF2                                                                                                                                                                               
if not hasattr(tf, 'logging'):                                                                                                                                                                               
  try:                                                                                                                                                                                                     
      tf.logging = tf.compat.v1.logging                                                                                                                                                                    
  except AttributeError:                                                                                                                                                                                   
      import logging as _pylog, types as _types                                                                                                                                                            
      tf.logging = _types.SimpleNamespace(                                                                                                                                                                 
          info=_pylog.info, warning=_pylog.warning,                                                                                                                                                        
          error=_pylog.error, debug=_pylog.debug,                                                                                                                                                          
          fatal=_pylog.critical,                                                                                                                                                                           
      )                                                                                                                                                                                                    
  print('tf.logging compat patch applied.')                                                                                                                                                                
                                                                                                                                                                                                           
# 2. ResNet layers= kwarg removed in Keras >= 2.13                                                                                                                                                           
def _make_resnet_compat(orig_fn):                                                                                                                                                                            
  def _compat(*args, **kwargs):                                                                                                                                                                            
      try:                                                                                                                                                                                                 
          return orig_fn(*args, **kwargs)                                                                                                                                                                  
      except TypeError:                                                                                                                                                                                    
          kwargs.pop('layers', None)                                                                                                                                                                      
          return orig_fn(*args, **kwargs)                                                                                                                                                                  
  return _compat                                                                                                                                                                                           
                                                                                                                                                                                                           
for _rname in ['ResNet50', 'ResNet101', 'ResNet152']:                                                                                                                                                        
  _orig = getattr(tf.keras.applications.resnet, _rname, None)                                                                                                                                              
  if _orig is not None:                                                                                                                                                                                    
      setattr(tf.keras.applications.resnet, _rname, _make_resnet_compat(_orig))                                                                                                                            
print('ResNet layers= compat patch applied.')                                                                                                                                                                
                                                                                                                                                                                                           
# 3. static_shape helpers receive tuples in Keras 3.x                                                                                                                                                        
try:                                                                                                                                                                                                         
  from object_detection.utils import static_shape as _ss                                                                                                                                                   
  def _make_ts_compat(fn):                                                                                                                                                                                 
      def _wrapped(tensor_shape):                                                                                                                                                                          
          if not hasattr(tensor_shape, 'assert_has_rank'):                                                                                                                                                 
              tensor_shape = tf.TensorShape(tensor_shape)                                                                                                                                                  
          return fn(tensor_shape)                                                                                                                                                                          
      return _wrapped                                                                                                                                                                                      
  for _fn_name in ['get_depth', 'get_height', 'get_width']:                                                                                                                                                
      _orig = getattr(_ss, _fn_name, None)                                                                                                                                                                 
      if _orig is not None:                                                                                                                                                                                
          setattr(_ss, _fn_name, _make_ts_compat(_orig))                                                                                                                                                   
  print('static_shape tuple-compat patch applied.')                                                                                                                                                        
except Exception as _e:                                                                                                                                                                                      
  print(f'  static_shape patch skipped: {_e}')                                                                                                                                                             
                                                                                                                                                                                                           
# 4. KerasTensor.experimental_ref removed in Keras 3                                                                                                                                                         
# Detect the actual KerasTensor class via tf.keras.Input, then add the method.                                                                                                                               
try:                                                                                                                                                                                                         
  _sample_kt = tf.keras.Input(shape=(1,))                                                                                                                                                                  
  _KT_cls = type(_sample_kt)                                                                                                                                                                               
  del _sample_kt                                                                                                                                                                                           
  if not hasattr(_KT_cls, 'experimental_ref'):                                                                                                                                                             
      _KT_REFS = {}                                                                                                                                                                                        
      def _kt_experimental_ref(self, _cache=_KT_REFS):                                                                                                                                                     
          k = id(self)                                                                                                                                                                                     
          _cache[k] = self  # keep alive to prevent id reuse                                                                                                                                               
          return k                                                                                                                                                                                         
      _KT_cls.experimental_ref = _kt_experimental_ref                                                                                                                                                      
      print(f'KerasTensor ({_KT_cls.__name__}).experimental_ref compat patch applied.')                                                                                                                    
except Exception as _e:                                                                                                                                                                                      
  print(f'  KerasTensor experimental_ref patch skipped: {_e}')                                                                                                                                             
                                                                                                                                                                                                           
# ── Chargement du modèle ───────────────────────────────────────────────                                                                                                                                    
MODEL_NAME = 'faster_rcnn_resnet50_v1_640x640_coco17_tpu-8'                                                                                                                                                  
MODEL_DATE = '20200711'                                                                                                                                                                                      
MODEL_URL = (f'http://download.tensorflow.org/models/object_detection/tf2/'                                                                                                                                  
           f'{MODEL_DATE}/{MODEL_NAME}.tar.gz')                                                                                                                                                            
CKPT_DIR = f'./{MODEL_NAME}'                                                                                                                                                                                 
                                                                                                                                                                                                           
if not os.path.exists(CKPT_DIR):                                                                                                                                                                             
  print(f'Downloading {MODEL_NAME} (~300 MB)...')                                                                                                                                                          
  urllib.request.urlretrieve(MODEL_URL, f'{MODEL_NAME}.tar.gz')                                                                                                                                            
  with tarfile.open(f'{MODEL_NAME}.tar.gz') as tar:                                                                                                                                         
      tar.extractall('.')                                                                                                                                                                                  
  os.remove(f'{MODEL_NAME}.tar.gz')                                                                                                                                                                        
  print('Extracted.')                                                                                                                                                                                      
else:                                                                                                                                                                                                        
  print(f'Model already at {CKPT_DIR}')                                                                                                                                                                    
                                                                                                                                                                                                           
configs = config_util.get_configs_from_pipeline_file(                                                                                                                                                        
  os.path.join(CKPT_DIR, 'pipeline.config')                                                                                                                                                                
)                                                                                                                                                                                                            
model_config = configs['model']                                                                                                                                                                              
model_config.faster_rcnn.num_classes = len(categories)                                                                                                                                                       
                                                                                                                                                                                                           
detection_model = model_builder.build(model_config=model_config, is_training=True)                                                                                                                           
ckpt = tf.train.Checkpoint(model=detection_model)                                                                                                                                                            
ckpt.restore(os.path.join(CKPT_DIR, 'checkpoint', 'ckpt-0')).expect_partial()                                                                                                                                
print('Faster R-CNN ResNet50 loaded.')                                                                                                                                                                       
                                                                                                                                                                                                           
# Warmup — en mode training, provide_groundtruth() est requis avant predict()                                                                                                                                
dummy        = tf.zeros([1, 640, 640, 3])                                                                                                                                                                    
pp, shapes_w = detection_model.preprocess(dummy)                                                                                                                                                             
detection_model.provide_groundtruth(                                                                                                                                                                         
  groundtruth_boxes_list=[tf.zeros([0, 4], dtype=tf.float32)],                                                                                                                                             
  groundtruth_classes_list=[tf.zeros([0, len(categories)], dtype=tf.float32)],                                                                                                                             
)                                                                                                                                                                                                            
detection_model.predict(pp, shapes_w)                                                                                                                                                                        
print(f'Model ready - {len(detection_model.trainable_variables)} trainable variables | '                                                                                                                     
    f'classes: {len(categories)}')      

Preparing TF Object Detection API (ModuleNotFoundError: No module named 'object_detection')


  Repo cloned.
Reading package lists...


Building dependency tree...


Reading state information...


The following additional packages will be installed:
  libprotoc23
Suggested packages:
  protobuf-mode-el
Recommended packages:
  libprotobuf-dev
The following packages will be upgraded:
  libprotoc23 protobuf-compiler
2 upgraded, 0 newly installed, 0 to remove and 131 not upgraded.
Need to get 692 kB of archives.
After this operation, 0 B of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 protobuf-compiler amd64 3.12.4-1ubuntu7.22.04.6 [29.2 kB]


...

I0000 00:00:1775344055.640245      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775344055.646711      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Faster R-CNN ResNet50 loaded.


I0000 00:00:1775344057.870260      23 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:depth of additional conv before box predictor: 0


Model ready - 222 trainable variables | classes: 4


In [10]:
from tensorflow.keras.optimizers import Adam

LEARNING_RATE = 1e-4
optimizer = Adam(learning_rate=LEARNING_RATE)
print(f"Optimizer: Adam, lr={LEARNING_RATE}")
# Faster R-CNN computes all losses internally via model.loss()

Optimizer: Adam, lr=0.0001


In [11]:
import numpy as np
import tensorflow as tf

IMG_SIZE = 320
NUM_CLASSES = len(categories)
cat_vals = list(categories.values())

_IMG_CACHE = {}  # filename → uint8 numpy HxWx3, populated before training

def prepare_batch(filenames_b, tiles_b, bboxes_b, classes_b):
    """
    Load and preprocess a batch for Faster R-CNN.

    Input bboxes: [xmin, ymin, xmax, ymax] in pixels (padded with 0, class=-1).
    Returns:
      images        : float32 [B, IMG_SIZE, IMG_SIZE, 3] kept in [0,255] (FRCNN preprocesses internally)
      gt_boxes_list : list of float32 [N_i, 4] normalized [ymin, xmin, ymax, xmax]  ← TF OD API format
      gt_classes_list: list of float32 [N_i, num_classes] one-hot encoded
    """
    images, gt_boxes_list, gt_classes_list = [], [], []
    for i in range(len(filenames_b)):
        fn      = filenames_b[i].decode('utf-8') if isinstance(filenames_b[i], bytes) else str(filenames_b[i])
        tile    = tiles_b[i]
        bboxes  = bboxes_b[i]
        cls_ids = classes_b[i]

        if fn in _IMG_CACHE:
            img = _IMG_CACHE[fn]
        else:
            img = load_geoimage(fn)
            img = np.array(tf.image.resize(tf.cast(img, tf.float32), [IMG_SIZE, IMG_SIZE]))
            _IMG_CACHE[fn] = img

        orig_h = float(tile[3])
        orig_w = float(tile[2])

        valid       = cls_ids >= 0
        valid_bboxes = bboxes[valid]
        valid_cls   = cls_ids[valid].astype(int)

        if len(valid_bboxes) > 0:
            # [xmin,ymin,xmax,ymax] pixels → [ymin,xmin,ymax,xmax] normalized
            norm_boxes = np.stack([
                valid_bboxes[:, 1] / orig_h,   # ymin
                valid_bboxes[:, 0] / orig_w,   # xmin
                valid_bboxes[:, 3] / orig_h,   # ymax
                valid_bboxes[:, 2] / orig_w,   # xmax
            ], axis=1).clip(0.0, 1.0).astype(np.float32)
            classes_oh = np.eye(NUM_CLASSES, dtype=np.float32)[valid_cls]
        else:
            norm_boxes = np.zeros((0, 4), dtype=np.float32)
            classes_oh = np.zeros((0, NUM_CLASSES), dtype=np.float32)

        images.append(img)
        gt_boxes_list.append(tf.constant(norm_boxes))
        gt_classes_list.append(tf.constant(classes_oh))

    return tf.stack(images, axis=0), gt_boxes_list, gt_classes_list

In [12]:
def xywh_to_xyxy(bb):
    x, y, w, h = bb
    return [x, y, x + w, y + h]

def extract_data(anns_list):
    filenames, tiles, all_bboxes, all_classes = [], [], [], []
    for img_ann in anns_list:
        filenames.append(img_ann.filename)
        tiles.append(list(img_ann.tile))
        img_bboxes, img_classes = [], []
        for obj in img_ann.objects:
            img_bboxes.append(xywh_to_xyxy(obj.bb))          # → [xmin,ymin,xmax,ymax]
            img_classes.append(cat_vals.index(obj.category))
        all_bboxes.append(img_bboxes)
        all_classes.append(img_classes)
    return filenames, tiles, all_bboxes, all_classes

filenames_train, tiles_train, bboxes_train, classes_train = extract_data(anns_train)
filenames_valid, tiles_valid, bboxes_valid, classes_valid = extract_data(anns_valid)

max_train = max((len(b) for b in bboxes_train), default=0)
max_valid = max((len(b) for b in bboxes_valid), default=0)
MAX_BOXES = min(300, max(1, max(max_train, max_valid)))
print(f"Max boxes per image: {MAX_BOXES}")

bboxes_train_padded  = [b[:MAX_BOXES] + [[0,0,0,0]] * (MAX_BOXES - min(len(b), MAX_BOXES)) for b in bboxes_train]
classes_train_padded = [c[:MAX_BOXES] + [-1]        * (MAX_BOXES - min(len(c), MAX_BOXES)) for c in classes_train]
bboxes_valid_padded  = [b[:MAX_BOXES] + [[0,0,0,0]] * (MAX_BOXES - min(len(b), MAX_BOXES)) for b in bboxes_valid]
classes_valid_padded = [c[:MAX_BOXES] + [-1]        * (MAX_BOXES - min(len(c), MAX_BOXES)) for c in classes_valid]

BATCH_SIZE = min(8, max(1, len(filenames_train)))

ds_train = tf.data.Dataset.from_tensor_slices((
    tf.constant(filenames_train),
    tf.constant(tiles_train,          dtype=tf.int32),
    tf.constant(bboxes_train_padded,  dtype=tf.float32),
    tf.constant(classes_train_padded, dtype=tf.int32)
)).shuffle(max(len(filenames_train), 1)).batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

ds_valid = tf.data.Dataset.from_tensor_slices((
    tf.constant(filenames_valid),
    tf.constant(tiles_valid,          dtype=tf.int32),
    tf.constant(bboxes_valid_padded,  dtype=tf.float32),
    tf.constant(classes_valid_padded, dtype=tf.int32)
)).batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

print(f"Train batches: {len(ds_train)}, Valid batches: {len(ds_valid)}, Batch size: {BATCH_SIZE}")
# ── Préchargement de toutes les images en RAM ──────────────────────────
import time as _time
all_fns = list(set(filenames_train + filenames_valid))
print(f"Preloading {len(all_fns)} images into RAM...")
_t0 = _time.time()
for _fn in all_fns:
    if _fn not in _IMG_CACHE:
        _img = load_geoimage(_fn)
        _IMG_CACHE[_fn] = np.array(tf.image.resize(tf.cast(_img, tf.float32), [IMG_SIZE, IMG_SIZE]))
print(f"  Done in {_time.time()-_t0:.1f}s — {len(_IMG_CACHE)} images cached.")


Max boxes per image: 300


Train batches: 2530, Valid batches: 191, Batch size: 4
Preloading 7606 images into RAM...


  Done in 97.9s — 7606 images cached.


In [13]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline

EPOCHS_PHASE1  = 2
EPOCHS_PHASE2  = 3
PATIENCE       = 2
WARMUP_EPOCHS  = 1
TOTAL_EPOCHS   = EPOCHS_PHASE1 + EPOCHS_PHASE2
MAX_STEPS_PER_EPOCH = 400  # cap steps per epoch to fit in Kaggle session (~20 min/epoch)
STEPS_PER_EPOCH = min(len(ds_train), MAX_STEPS_PER_EPOCH)
if STEPS_PER_EPOCH == 0:
    raise ValueError('Le dataset de train est vide apres batching.')

def get_lr(epoch, base_lr=1e-4, min_lr=1e-6):
    if epoch < WARMUP_EPOCHS:
        return base_lr * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(TOTAL_EPOCHS - WARMUP_EPOCHS, 1)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * progress))

all_vars      = detection_model.trainable_variables
backbone_vars = [v for v in all_vars if 'feature_extractor' in v.name.lower()]
head_vars     = [v for v in all_vars if 'feature_extractor' not in v.name.lower()]
freeze_until  = len(backbone_vars)  # freeze backbone entirely in phase 1
phase1_vars   = head_vars  # only train detection heads
phase2_vars   = list(all_vars)
if not phase1_vars:
    phase1_vars = list(all_vars)
print(f"Phase-1 trainable vars: {len(phase1_vars)} | Phase-2: {len(phase2_vars)}")

ckpt_train        = tf.train.Checkpoint(model=detection_model, optimizer=optimizer)
ckpt_manager      = tf.train.CheckpointManager(ckpt_train, './frcnn_ckpts', max_to_keep=3)
best_ckpt_manager = tf.train.CheckpointManager(
    tf.train.Checkpoint(model=detection_model), './frcnn_best', max_to_keep=1
)

def run_train_step(images, gt_boxes_list, gt_classes_list, trainable_vars):
    preprocessed, shapes = detection_model.preprocess(images)
    detection_model.provide_groundtruth(
        groundtruth_boxes_list=gt_boxes_list,
        groundtruth_classes_list=gt_classes_list
    )
    with tf.GradientTape() as tape:
        pred   = detection_model.predict(preprocessed, shapes)
        losses = detection_model.loss(pred, shapes)
        total  = tf.add_n(list(losses.values()))
    grads = tape.gradient(total, trainable_vars)
    grads_and_vars = [(g, v) for g, v in zip(grads, trainable_vars) if g is not None]
    optimizer.apply_gradients(grads_and_vars)
    return float(total), {k: float(v) for k, v in losses.items()}

def run_val_step(images, gt_boxes_list, gt_classes_list):
    preprocessed, shapes = detection_model.preprocess(images)
    detection_model.provide_groundtruth(
        groundtruth_boxes_list=gt_boxes_list,
        groundtruth_classes_list=gt_classes_list
    )
    pred   = detection_model.predict(preprocessed, shapes)
    losses = detection_model.loss(pred, shapes)
    total  = tf.add_n(list(losses.values()))
    return float(total), {k: float(v) for k, v in losses.items()}

history = {k: [] for k in [
    'train_total', 'val_total',
    'train_rpn_loc', 'train_rpn_obj', 'val_rpn_loc', 'val_rpn_obj',
    'train_cls_loc', 'train_cls_cls', 'val_cls_loc', 'val_cls_cls',
]}
best_val_loss    = float('inf')
patience_counter = 0

def _get(loss_dict, key, default=0.0):
    for k, v in loss_dict.items():
        if key in k.lower():
            return v
    return default

for epoch in range(TOTAL_EPOCHS):
    phase = 1 if epoch < EPOCHS_PHASE1 else 2
    current_vars = phase1_vars if epoch < EPOCHS_PHASE1 else phase2_vars

    if epoch == 0:
        print(f"Phase 1 - {len(phase1_vars)} trainable vars (backbone partially frozen)")
    if epoch == EPOCHS_PHASE1:
        print(f"\nPhase 2 - {len(phase2_vars)} trainable vars (full fine-tuning)")

    lr = get_lr(epoch)
    try:
        optimizer.learning_rate.assign(lr)
    except Exception:
        tf.keras.backend.set_value(optimizer.learning_rate, lr)

    t_total, t_rpn_loc, t_rpn_obj, t_cls_loc, t_cls_cls = [], [], [], [], []
    for step, batch in enumerate(ds_train):
        if step >= MAX_STEPS_PER_EPOCH:
            break
        fns_b, tiles_b, bboxes_b, cls_b = batch
        imgs, boxes_list, classes_list = prepare_batch(
            fns_b.numpy(), tiles_b.numpy(), bboxes_b.numpy(), cls_b.numpy()
        )
        total_loss, ld = run_train_step(imgs, boxes_list, classes_list, current_vars)
        t_total.append(total_loss)
        t_rpn_loc.append(_get(ld, 'rpn_localization'))
        t_rpn_obj.append(_get(ld, 'rpn_objectness'))
        t_cls_loc.append(_get(ld, 'localization'))
        t_cls_cls.append(_get(ld, 'classification'))
        if (step + 1) % 100 == 0:
            print(f"  [{epoch+1}/{TOTAL_EPOCHS}] step {step+1}/{STEPS_PER_EPOCH} loss={np.mean(t_total):.4f} lr={lr:.2e}")

    history['train_total'].append(float(np.mean(t_total)))
    history['train_rpn_loc'].append(float(np.mean(t_rpn_loc)))
    history['train_rpn_obj'].append(float(np.mean(t_rpn_obj)))
    history['train_cls_loc'].append(float(np.mean(t_cls_loc)))
    history['train_cls_cls'].append(float(np.mean(t_cls_cls)))

    v_total, v_rpn_loc, v_rpn_obj, v_cls_loc, v_cls_cls = [], [], [], [], []
    for v_step, batch in enumerate(ds_valid):
        if v_step >= 100:
            break
        fns_b, tiles_b, bboxes_b, cls_b = batch
        imgs, boxes_list, classes_list = prepare_batch(
            fns_b.numpy(), tiles_b.numpy(), bboxes_b.numpy(), cls_b.numpy()
        )
        total_loss, ld = run_val_step(imgs, boxes_list, classes_list)
        v_total.append(total_loss)
        v_rpn_loc.append(_get(ld, 'rpn_localization'))
        v_rpn_obj.append(_get(ld, 'rpn_objectness'))
        v_cls_loc.append(_get(ld, 'localization'))
        v_cls_cls.append(_get(ld, 'classification'))

    val_mean = float(np.mean(v_total)) if v_total else history['train_total'][-1]
    history['val_total'].append(val_mean)
    history['val_rpn_loc'].append(float(np.mean(v_rpn_loc)) if v_rpn_loc else 0.0)
    history['val_rpn_obj'].append(float(np.mean(v_rpn_obj)) if v_rpn_obj else 0.0)
    history['val_cls_loc'].append(float(np.mean(v_cls_loc)) if v_cls_loc else 0.0)
    history['val_cls_cls'].append(float(np.mean(v_cls_cls)) if v_cls_cls else 0.0)

    print(f"Epoch {epoch+1}/{TOTAL_EPOCHS} | train={history['train_total'][-1]:.4f} val={val_mean:.4f} lr={lr:.2e} phase={phase}")

    ckpt_manager.save()
    if val_mean < best_val_loss:
        best_val_loss = val_mean
        patience_counter = 0
        best_ckpt_manager.save()
        print(f"  Best model saved (val={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

ep = list(range(1, len(history['train_total']) + 1))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(ep, history['train_total'], label='Train')
axes[0].plot(ep, history['val_total'], label='Val')
axes[0].axvline(EPOCHS_PHASE1, color='gray', ls='--', alpha=.6, label='Phase 2')
axes[0].set_title('Total Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(ep, history['train_rpn_loc'], label='Train loc')
axes[1].plot(ep, history['val_rpn_loc'], label='Val loc')
axes[1].plot(ep, history['train_rpn_obj'], label='Train obj', ls='--')
axes[1].plot(ep, history['val_rpn_obj'], label='Val obj', ls='--')
axes[1].set_title('RPN Losses'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(ep, history['train_cls_loc'], label='Train loc')
axes[2].plot(ep, history['val_cls_loc'], label='Val loc')
axes[2].plot(ep, history['train_cls_cls'], label='Train cls', ls='--')
axes[2].plot(ep, history['val_cls_cls'], label='Val cls', ls='--')
axes[2].set_title('Classifier Losses'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig('frcnn_training_curves.png', dpi=150)
plt.show()
print(f"Best val_loss: {best_val_loss:.4f}")

lr_vals = [get_lr(e) for e in range(TOTAL_EPOCHS)]
fig2, ax2 = plt.subplots(figsize=(8, 3))
ax2.plot(range(1, TOTAL_EPOCHS + 1), lr_vals, color='steelblue')
ax2.axvspan(1, WARMUP_EPOCHS, alpha=.15, color='orange', label='Warmup')
ax2.axvspan(WARMUP_EPOCHS + 1, TOTAL_EPOCHS, alpha=.10, color='green', label='Cosine decay')
ax2.axvline(EPOCHS_PHASE1, color='gray', ls='--', alpha=.6, label='Phase 2')
ax2.set_title('LR Schedule'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR')
ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

Phase-1 trainable vars: 222 | Phase-2: 222
Phase 1 - 222 trainable vars (backbone partially frozen)


Instructions for updating:

Future major versions of TensorFlow will allow gradients to flow
into the labels input on backprop by default.

See `tf.nn.softmax_cross_entropy_with_logits_v2`.



  [1/5] step 100/2530 loss=4.2994 lr=1.00e-04


  [1/5] step 200/2530 loss=3.6984 lr=1.00e-04


  [1/5] step 300/2530 loss=3.4944 lr=1.00e-04


...

  [5/5] step 1000/2530 loss=2.1940 lr=1.55e-05


  [5/5] step 1100/2530 loss=2.1911 lr=1.55e-05


  [5/5] step 1200/2530 loss=2.1936 lr=1.55e-05


  [5/5] step 1300/2530 loss=2.1939 lr=1.55e-05


  [5/5] step 1400/2530 loss=2.1937 lr=1.55e-05


#### Validation
Compute validation metrics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as col
import numpy as np
%matplotlib inline

def area_intersection(boxes, box):
    xmin = np.maximum(np.min(boxes[:, 0::2], axis=1), np.min(box[0::2]))
    ymin = np.maximum(np.min(boxes[:, 1::2], axis=1), np.min(box[1::2]))
    xmax = np.minimum(np.max(boxes[:, 0::2], axis=1), np.max(box[0::2]))
    ymax = np.minimum(np.max(boxes[:, 1::2], axis=1), np.max(box[1::2]))
    w = np.maximum(xmax - xmin + 1.0, 0.0)
    h = np.maximum(ymax - ymin + 1.0, 0.0)
    return w * h

def area_union(boxes, box):
    area_anns = (np.max(box[0::2])-np.min(box[0::2])+1.0) * (np.max(box[1::2])-np.min(box[1::2])+1.0)
    area_pred = (np.max(boxes[:, 0::2], axis=1)-np.min(boxes[:, 0::2], axis=1)+1.0) * (np.max(boxes[:, 1::2], axis=1)-np.min(boxes[:, 1::2], axis=1)+1.0)
    return area_anns + area_pred - area_intersection(boxes, box)

def calc_iou(boxes, box):
    iou = area_intersection(boxes, box) / area_union(boxes, box)
    max_value = np.max(iou)
    max_index = np.argmax(iou)
    return max_value, max_index

def calc_ap(rec, prec):
    # First append sentinel values at the end
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    # Compute the precision envelope
    for i in range(mpre.size-1, 0, -1):
        mpre[i-1] = np.maximum(mpre[i-1], mpre[i])
    # To calculate area under PR curve, look for points where X axis (recall) changes value
    i = np.where(mrec[1:] != mrec[:-1])[0]
    # and sum (\Delta recall) * prec
    ap = np.sum((mrec[i+1] - mrec[i]) * mpre[i+1])
    return ap

def draw_confusion_matrix(cm, categories):
    # Draw confusion matrix
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    cm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], np.finfo(np.float64).eps)
    im = ax.imshow(cm, interpolation='nearest', cmap=getattr(plt, 'colormaps', plt.cm)['Blues'] if hasattr(plt, 'colormaps') else plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), xticklabels=categories, yticklabels=categories, ylabel='Annotation', xlabel='Prediction')
    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    # Loop over data dimensions and create text annotations
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], '.2f'), ha="center", va="center", color="white" if cm[i, j] > thresh else "black", fontsize=int(20-pow(len(categories), 0.5)))
    fig.tight_layout()
    plt.show()

def draw_precision_recall(precisions, recalls, categories):
    # Draw precision-recall curves for each category
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    plt.axis([0, 1, 0, 1])
    c_dark = list(filter(lambda x: x.startswith('dark'), col.cnames.keys()))
    aps = []
    # Compare categories for a specific algorithm
    for idx in range(len(categories)):
        plt.plot(recalls[idx], precisions[idx], color=c_dark[idx], label=categories[idx], linewidth=4.0)
        aps.append(calc_ap(recalls[idx], precisions[idx]))
    handles, labels = ax.get_legend_handles_labels()
    labels = [str(val + ' [' + "{:.3f}".format(aps[idx]) + ']') for idx, val in enumerate(labels)]
    handles = [h for (ap, h) in sorted(zip(aps, handles), key=lambda x: x[0], reverse=True)]
    labels = [l for (ap, l) in sorted(zip(aps, labels), key=lambda x: x[0], reverse=True)]
    leg = plt.legend(handles, labels, loc='upper right')
    leg.set_zorder(100)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.grid("on", linestyle="--", linewidth=2.0)
    fig.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from tqdm import tqdm

# ── Restore best checkpoint ──────────────────────────────────────────
infer_model = detection_model
best_ckpt_path = tf.train.latest_checkpoint('./frcnn_best')
if best_ckpt_path:
    best_ckpt_restore = tf.train.Checkpoint(model=infer_model)
    best_ckpt_restore.restore(best_ckpt_path).expect_partial()
    print(f"Best Faster R-CNN weights restored from {best_ckpt_path}")
else:
    print("No best checkpoint found in ./frcnn_best, using current model weights.")

CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45

annotations, predictions = {}, {}

for ann in tqdm(anns_valid, desc='Validating'):
    fn = ann.filename
    annotations.setdefault(fn, {})
    predictions.setdefault(fn, {})

    # Ground truth
    for obj in ann.objects:
        cat = obj.category
        annotations[fn].setdefault(cat, {'bbox': []})
        annotations[fn][cat]['bbox'].append(xywh_to_xyxy(obj.bb))

    # Load image
    img = load_geoimage(fn)
    h_orig = float(ann.tile[3]) if ann.tile[3] > 0 else float(img.shape[0])
    w_orig = float(ann.tile[2]) if ann.tile[2] > 0 else float(img.shape[1])
    img_resized = np.array(
        tf.image.resize(tf.cast(img, tf.float32), [IMG_SIZE, IMG_SIZE])
    )[np.newaxis]   # [1, H, W, 3]

    # Inference (switch model to eval mode: no GradientTape, use postprocess)
    preprocessed, shapes = infer_model.preprocess(tf.constant(img_resized))
    pred_dict = infer_model.predict(preprocessed, shapes)
    detections = infer_model.postprocess(pred_dict, shapes)

    # Parse outputs — TF OD API returns [ymin, xmin, ymax, xmax] normalized
    boxes_norm  = detections['detection_boxes'][0].numpy()          # [N, 4]
    scores      = detections['detection_scores'][0].numpy()         # [N]
    class_ids   = detections['detection_classes'][0].numpy().astype(int)  # [N]  1-indexed

    for i in range(len(scores)):
        if scores[i] < CONF_THRESHOLD:
            continue
        ymin, xmin, ymax, xmax = boxes_norm[i]
        # Back to pixel [xmin, ymin, xmax, ymax]
        x1 = xmin * w_orig; y1 = ymin * h_orig
        x2 = xmax * w_orig; y2 = ymax * h_orig
        # TF OD API class indices start at 1 for custom classes
        cat_idx = class_ids[i] - 1
        if cat_idx < 0 or cat_idx >= NUM_CLASSES:
            continue
        cat = cat_vals[cat_idx]
        predictions[fn].setdefault(cat, {'bbox': [], 'confidence': []})
        predictions[fn][cat]['bbox'].append([x1, y1, x2, y2])
        predictions[fn][cat]['confidence'].append(float(scores[i]))

In [ ]:
threshold = 0.5
default_cls = 'BACKGROUND'
y_true, y_pred = [], []  # confusion matrix
tps, confidences = dict(), dict()  # draw precision-recall curves for each category
for cls in categories.values():
    # Compute TP, FP and FN for each image
    tps[cls], confidences[cls] = [], []
    for f in predictions:
        # Sort 'cls' predictions by confidence for each file
        pred_boxes, pred_confidences = [], []
        if cls in predictions[f].keys():
            for idx in range(len(predictions[f][cls]['bbox'])):
                pred_boxes.append(predictions[f][cls]['bbox'][idx])
                pred_confidences.append(predictions[f][cls]['confidence'][idx])
            sorted_ind = np.argsort(-np.array(pred_confidences))
            pred_boxes = np.array(pred_boxes)[sorted_ind, :]
        pred_boxes = np.array(pred_boxes).astype(float)
        # Define 'cls' annotations for each file
        anno_boxes = []
        if cls in annotations[f].keys():
            anno_boxes = annotations[f][cls]['bbox']
        anno_boxes = np.array(anno_boxes).astype(float)
        # Define horizontal or oriented bounding boxes
        anno_indices = list(range(len(anno_boxes)))
        # Compare a single prediction 'pred_box' with all annotations 'anno_boxes'
        for pred_idx, pred_box in enumerate(pred_boxes):
            # A prediction is correct if its IoU with the ground truth is above the threshold
            iou_value, ann_index = calc_iou(anno_boxes, pred_box) if len(anno_boxes) > 0 else (-1, -1)
            if iou_value > threshold and ann_index in anno_indices:
                # TP
                anno_indices.remove(int(ann_index))
                tps[cls] += [1.0]
                y_true += [cls]
            else:
                # FP
                tps[cls] += [0.0]
                y_true += [default_cls]
            y_pred += [cls]
            confidences[cls] += [pred_confidences[pred_idx]]
        # FN
        y_true += [cls] * len(anno_indices)
        y_pred += [default_cls] * len(anno_indices)
y_true, y_pred = np.array(y_true), np.array(y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score

# Compute AP metric
precision_list, recall_list, ap_list = [], [], []
for cls in categories.values():
    sorted_ind = np.argsort(-np.array(confidences[cls]))
    tp = np.cumsum(np.array(tps[cls])[sorted_ind], dtype=float)
    recall = np.array([0.0]) if len(tp) == 0 else tp / np.maximum(np.sum(y_true == cls), np.finfo(np.float64).eps)
    precision = np.array([0.0]) if len(tp) == 0 else tp / np.maximum(list(range(1, np.sum(y_pred == cls)+1)), np.finfo(np.float64).eps)
    ap = calc_ap(recall, precision)
    print('> %s: Recall: %.3f%% Precision: %.3f%% AP: %.3f%%' % (cls, recall[-1]*100, precision[-1]*100, ap*100))
    precision_list.append(precision)
    recall_list.append(recall)
    ap_list.append(ap)
mean_ap = np.mean(ap_list)
print('mAccuracy: %.3f%%' % (accuracy_score(y_true, y_pred)*100))
print('mRecall: %.3f%%' % (recall_score(y_true, y_pred, average='macro', zero_division=1)*100))
print('mPrecision: %.3f%%' % (precision_score(y_true, y_pred, average='macro', zero_division=1)*100))
print('mAP: %.3f%%' % (mean_ap*100))

In [ ]:
names = list(categories.values()).copy()
names.insert(0, default_cls)
cm = confusion_matrix(y_true, y_pred, labels=names)
print('Confusion matrix:')
print(cm)
draw_confusion_matrix(cm, names)
draw_precision_recall(precision_list, recall_list, categories)

#### Testing
Try to improve the results provided in the competition.

In [ ]:
import os
import numpy as np

test_dir = None
search_roots = [BASE_DIR, '.', '..', '../..']
for search_root in search_roots:
    if not search_root or not os.path.exists(search_root):
        continue
    for root, dirs, files in os.walk(search_root):
        if 'xview_test' in dirs:
            test_dir = os.path.abspath(os.path.join(root, 'xview_test'))
            break
    if test_dir:
        break

if test_dir is None:
    raise FileNotFoundError("Dossier xview_test introuvable. Verifie l'extraction du dataset.")

anns = []
for (dirpath, dirnames, filenames) in os.walk(test_dir):
    for filename in filenames:
        full_path = os.path.abspath(os.path.join(dirpath, filename))
        image = GenericImage(full_path)
        image.tile = np.array([-1, -1, -1, -1])
        anns.append(image)
print('Number of testing images: ' + str(len(anns)))

In [ ]:
import os
import numpy as np
from tqdm import tqdm
import json

best_ckpt_path2 = tf.train.latest_checkpoint('./frcnn_best')
if best_ckpt_path2:
    best_ckpt_restore2 = tf.train.Checkpoint(model=detection_model)
    best_ckpt_restore2.restore(best_ckpt_path2).expect_partial()
    print(f"Best weights restored for test inference from {best_ckpt_path2}.")
else:
    print("No best checkpoint found for test inference, using current model weights.")

CONF_THRESHOLD = 0.25

predictions_test = {}
predictions_data = {"images": {}, "annotations": {}, "categories": categories}
imgs_idx, annos_idx = 0, 0

for ann in tqdm(anns, desc='Testing'):
    fn = ann.filename
    predictions_test.setdefault(fn, {})

    img = load_geoimage(fn)
    h_orig = float(ann.tile[3]) if ann.tile[3] > 0 else float(img.shape[0])
    w_orig = float(ann.tile[2]) if ann.tile[2] > 0 else float(img.shape[1])

    img_resized = np.array(
        tf.image.resize(tf.cast(img, tf.float32), [IMG_SIZE, IMG_SIZE])
    )[np.newaxis]

    preprocessed, shapes = detection_model.preprocess(tf.constant(img_resized))
    pred_dict  = detection_model.predict(preprocessed, shapes)
    detections = detection_model.postprocess(pred_dict, shapes)

    boxes_norm = detections['detection_boxes'][0].numpy()
    scores     = detections['detection_scores'][0].numpy()
    class_ids  = detections['detection_classes'][0].numpy().astype(int)

    num_objects = 0
    for i in range(len(scores)):
        if scores[i] < CONF_THRESHOLD:
            continue
        ymin, xmin, ymax, xmax = boxes_norm[i]
        x1 = int(xmin * w_orig); y1 = int(ymin * h_orig)
        x2 = int(xmax * w_orig); y2 = int(ymax * h_orig)
        cat_idx = class_ids[i] - 1
        if cat_idx < 0 or cat_idx >= NUM_CLASSES:
            continue
        cat = cat_vals[cat_idx]
        predictions_test[fn].setdefault(cat, {'bbox': [], 'confidence': []})
        predictions_test[fn][cat]['bbox'].append([x1, y1, x2, y2])
        predictions_test[fn][cat]['confidence'].append(float(scores[i]))

        predictions_data["annotations"][annos_idx] = {
            "image_id": os.path.basename(fn),
            "category_id": cat,
            "bbox": (x1, y1, x2 - x1, y2 - y1),   # xywh for submission
            "confidence": str(scores[i])
        }
        annos_idx  += 1
        num_objects += 1

    predictions_data["images"][imgs_idx] = {
        "image_id": os.path.basename(fn),
        "filename": fn,
        "num_objects": num_objects,
        "width": int(w_orig), "height": int(h_orig)
    }
    imgs_idx += 1

with open("prediction_frcnn.json", "w") as outfile:
    json.dump(predictions_data, outfile)

print(f"Saved: {annos_idx} detections on {imgs_idx} images -> prediction_frcnn.json")